In [0]:
import pandas as pd
from pyspark.sql import SparkSession

In [0]:
import re
import os
import requests
from datetime import datetime

# GitHub repository configuration and target Databricks Bronze volume base path
github_owner = "Souravroyy-srv"
github_repo = "latest_data"
github_branch = "main"

bronze_base_path = "/Volumes/severn_trent/bronze/raw_data"

# Read repository root and find folders such as 01_04_2026
repository_url = (
    f"https://api.github.com/repos/{github_owner}/{github_repo}/contents"
    f"?ref={github_branch}"
)

response = requests.get(repository_url)
response.raise_for_status()

repository_items = response.json()

# Filter items to retain only directories matching the DD_MM_YYYY regex pattern
snapshot_folders = [
    item["name"]
    for item in repository_items
    if item["type"] == "dir"
    and re.fullmatch(r"\d{2}_\d{2}_\d{4}", item["name"])
]

if not snapshot_folders:
    raise ValueError("No snapshot folders were found in the GitHub repository.")

# Converts DD_MM_YYYY folder names to dates and selects the newest one
latest_snapshot = max(
    snapshot_folders,
    key=lambda folder: datetime.strptime(folder, "%d_%m_%Y")
)

print(f"Latest snapshot folder found: {latest_snapshot}")

# Read files from only the newest GitHub folder
folder_url = (
    f"https://api.github.com/repos/{github_owner}/{github_repo}/contents/"
    f"{latest_snapshot}?ref={github_branch}"
)

folder_response = requests.get(folder_url)
folder_response.raise_for_status()

folder_items = folder_response.json()

# Create matching snapshot folder in the Bronze Volume
target_snapshot_path = f"{bronze_base_path}/{latest_snapshot}"
os.makedirs(target_snapshot_path, exist_ok=True)

# Iterate through folder contents, download CSV files, and write them to the Bronze volume path
for item in folder_items:
    if item["type"] != "file" or not item["name"].lower().endswith(".csv"):
        continue

    file_name = item["name"]
    source_url = item["download_url"]
    target_file_path = f"{target_snapshot_path}/{file_name}"

    print(f"Copying: {file_name}")

    source_response = requests.get(source_url)
    source_response.raise_for_status()

    # Writes the original CSV file unchanged
    with open(target_file_path, "wb") as target_file:
        target_file.write(source_response.content)

print(f"Latest snapshot {latest_snapshot} ingested successfully.")